# Notebook 02: NMS and mAP Evaluation

**Module:** 08 Object Detection  
**Duration:** ~2.5 hrs

---

## Learning Objectives

1. Implement Non-Maximum Suppression from scratch
2. Understand precision-recall curves
3. Define mAP@0.5 and mAP@0.5:0.95
4. Know COCO evaluation protocol

## Non-Maximum Suppression (NMS)

Problem: detector outputs many overlapping boxes for one object.

**Algorithm:**
1. Sort boxes by confidence (descending)
2. Keep highest-scoring box
3. Remove boxes with IoU > threshold to kept box
4. Repeat

Typical IoU threshold: 0.5

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
import torch.nn.functional as F

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
rng = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
def box_iou(boxes1, boxes2):
    """boxes: (N,4) and (M,4) in xyxy format. Returns (N,M)."""
    x1 = torch.max(boxes1[:, None, 0], boxes2[None, :, 0])
    y1 = torch.max(boxes1[:, None, 1], boxes2[None, :, 1])
    x2 = torch.min(boxes1[:, None, 2], boxes2[None, :, 2])
    y2 = torch.min(boxes1[:, None, 3], boxes2[None, :, 3])
    inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    union = area1[:, None] + area2[None, :] - inter
    return inter / (union + 1e-7)


def xywh_to_xyxy(boxes):
    x, y, w, h = boxes.unbind(-1)
    return torch.stack([x, y, x + w, y + h], dim=-1)


def xyxy_to_cxcywh(boxes):
    x1, y1, x2, y2 = boxes.unbind(-1)
    return torch.stack([(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1], dim=-1)

In [ ]:
def nms(boxes, scores, iou_thresh=0.5):
    """Greedy NMS. boxes: (N,4) xyxy, scores: (N,)."""
    order = scores.argsort(descending=True)
    keep = []
    while order.numel() > 0:
        i = order[0].item()
        keep.append(i)
        if order.numel() == 1:
            break
        ious = box_iou(boxes[i:i+1], boxes[order[1:]])[0]
        mask = ious <= iou_thresh
        order = order[1:][mask]
    return torch.tensor(keep, dtype=torch.long)

In [ ]:
# NMS demo: 5 overlapping boxes, one true object
boxes = torch.tensor([
    [10., 10., 50., 50.],
    [12., 12., 52., 52.],
    [15., 8., 48., 49.],
    [200., 200., 240., 240.],
    [202., 198., 242., 238.],
])
scores = torch.tensor([0.92, 0.88, 0.75, 0.85, 0.80])
keep = nms(boxes, scores, iou_thresh=0.5)
print('Kept indices:', keep.tolist())
print('Kept boxes:', boxes[keep].tolist())

## mAP (mean Average Precision)

1. Match predictions to GT by IoU threshold (e.g. 0.5)
2. Sort by confidence → precision-recall curve
3. **AP** = area under PR curve (per class)
4. **mAP** = mean AP across classes

**COCO:** mAP@0.5:0.95 = average AP at IoU 0.5, 0.55, ..., 0.95

| Metric | Meaning |
|--------|--------|
| mAP@0.5 | Loose matching (PASCAL VOC style) |
| mAP@0.5:0.95 | Strict COCO benchmark |

In [ ]:
# Simple AP intuition: 10 GT, sorted predictions
# TP if IoU>=0.5 and unmatched GT; else FP
precisions = [1.0, 1.0, 0.67, 0.75, 0.8]
recalls = [0.1, 0.2, 0.3, 0.4, 0.5]
plt.plot(recalls, precisions, 'o-'); plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall curve (concept)'); plt.grid(True); plt.show()

---

## Summary

NMS removes duplicate detections; mAP is the standard detection metric.

**Next:** [03_Detection_Loss_Functions.ipynb](03_Detection_Loss_Functions.ipynb)